# 05 — Faithfulness, By Hand

Retrieval can be perfect, the chunk can be clean, and the answer can *still* be wrong in the one way that matters most: it can say something the context never actually supports. That's **faithfulness** -- not "is the answer correct" (that's notebook `06`), but "is every claim in this answer actually backed by what's in front of the model."

The mechanical version of this check is simple to state: extract each claim, decide if the context supports it, done. The two worked examples below exist because real unfaithfulness rarely looks like a claim invented from nothing. It looks like something much sneakier.


In [ ]:
import json
import os

DATA_DIR = os.path.join("..", "data")
with open(os.path.join(DATA_DIR, "eval_examples.json")) as f:
    examples = json.load(f)


## Worked example 1 — not fabrication, inversion

Read the context closely before reading either answer. This one has a specific relationship stated between two names -- the question asks about that exact relationship, backwards.


In [ ]:
ex = examples[17]
print("Question:", ex["question"])
print()
print("Context:", ex["context_text"])
print()
print("Gold:", ex["gold_answer"])
print("Honest:", ex["system_answer"])
print("Overconfident:", ex["system_answer_overconfident"])


**Judgment:** the honest answer is faithful -- the context never states who Charles III swore fealty to (it states the opposite direction: Rollo swore fealty *to* Charles III), so "the context does not contain information about this" is correct. The overconfident answer, "King Charles III swore fealty to Rollo," is unfaithful -- and notice it isn't a claim invented from nothing. The context genuinely contains a fealty relationship between these two names. The overconfident answer just **reversed the direction** and stated it as fact.

This is worth sitting with because it's a much easier failure to miss than a claim pulled from thin air. If you're only checking "does this name/relationship appear in the context," the answer passes -- Rollo and Charles III and "fealty" are all right there. Faithfulness has to check the *specific direction and shape* of the claim, not just whether its ingredients are present.


## Worked example 2 — a claim that quietly changes scope

Second trap: read exactly one word in the question against exactly one word in the context.


In [ ]:
ex = examples[19]
print("Question:", ex["question"])
print()
print("Context:", ex["context_text"][:300], "...")
print()
print("Gold:", ex["gold_answer"])
print("Honest:", ex["system_answer"])


**Judgment:** look closely -- the context says the Norman dynasty had a major impact on **medieval** Europe. The question asks about impact on **modern** Europe. The "honest" system answer says: "a major political, cultural, and military impact on **modern** Europe" -- lifting the three-item list straight from the context, but silently swapping "medieval" for "modern" to match the question's own wording. That's unfaithful, even though every individual word in the answer appears somewhere in the context. The *claim* -- "this had that impact on modern Europe" -- was never actually stated. Only "medieval Europe" was.

This is the subtlest kind of unfaithfulness: not invention, not inversion, just quietly matching the question's framing instead of the context's actual scope. Claim-level faithfulness checking has to catch this, and a check that only asks "do these words appear in the context" will sail right past it, exactly the way it's easy to on a first read.


Now look at the overconfident version of the same example (`examples[19]["system_answer_overconfident"]`) -- multiple paragraphs inventing feudal law's influence on "the modern nation-state," Norman French shaping "modern lexicon," and military tactics underpinning "contemporary European states." None of that specificity is in the context at all. This one *is* fabrication from nothing, sitting right next to worked example 1's inversion and this example's own scope-shift -- three different shapes of the same underlying failure, all in one small dataset.


## Your turn: claim-by-claim on the rest

For every example, break the answer into individual claims, and mark each one supported or unsupported. Pay the same close attention worked examples 1 and 2 needed -- word overlap with the context is not the same thing as the claim actually being stated.


In [ ]:
for i in range(20):
    ex = examples[i]
    print(f"[{i}] ({ex['source']}) Q: {ex['question']}")
    print(f"    Answer: {ex['system_answer'][:200]}")
    if "system_answer_overconfident" in ex:
        print(f"    Overconfident: {ex['system_answer_overconfident'][:200]}")
    print()


In [ ]:
faithfulness_notes = {
    17: [{"claim": "Charles III swore fealty to Rollo", "supported": False, "note": "inverts the actual stated relationship"}],
    19: [
        {"claim": "Political/cultural/military impact", "supported": True, "note": "directly stated"},
        {"claim": "...on modern Europe", "supported": False, "note": "context says medieval Europe, not modern"},
    ],
    # Extract and judge the claims for every other index, especially 15, 16, 18
    # (the remaining unanswerable questions with an overconfident variant to compare against):
    # 15: [{"claim": "...", "supported": True}],
}

print(f"Claims recorded for {len(faithfulness_notes)}/20 examples")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Faithfulness | Whether every claim in an answer is actually supported by the given context |
| Claim extraction | Breaking an answer into individual factual statements before judging each one |
| Inversion | A claim that reverses a real relationship stated in the context -- not invented, just backwards |
| Scope shift | A claim that quietly generalizes or reframes something the context stated more narrowly |

Three distinct ways an answer can be unfaithful showed up in just two worked examples: outright fabrication (nothing in the context resembles the claim at all), inversion (the ingredients are real, the direction is flipped), and scope-shift (the words match, the actual scope doesn't). A faithfulness check that only asks "do these words appear somewhere in the context" catches the first and misses the other two completely.

**Next up:** notebook `06` asks the question faithfulness alone can't answer -- an answer can be perfectly faithful to its context and still be the wrong answer, if the context itself doesn't fully cover what was asked, or the answer is faithful but incomplete.

**Exercises:**

- Finish `faithfulness_notes` for the remaining 18 examples, paying special attention to the CUAD examples (0-9) where legal language makes scope-shifts easy to miss.
- Find one more inversion-style unfaithfulness case among the 20, if one exists, following worked example 1's pattern.
- For example 19's overconfident answer, list every specific claim (there are several) and check each one individually -- are they *all* unsupported, or does one accidentally happen to be true anyway?
